### Методы динамического программирования

#### Итерация по ценностям (Value Iteration)

```python
import numpy as np

def value_iteration(P, R, gamma, epsilon=1e-4):
    """
    Итерация по ценностям (Value Iteration) для конечного МППР.
    
    Параметры:
    - P: np.ndarray формы (|S|, |A|, |S|) — вероятности переходов P[s, a, s'] = p(s' | s, a)
    - R: np.ndarray формы (|S|, |A|) — ожидаемая награда R[s, a] = R(s, a)
    - gamma: float — коэффициент дисконтирования
    - epsilon: float — порог остановки по критерию сходимости
    
    Возвращает:
    - V: np.ndarray длины |S| — приближение оптимальной функции ценности V*
    - pi: np.ndarray длины |S| — оптимальная детерминированная стратегия \pi*(s)
    """
    n_states, n_actions, _ = P.shape
    # Инициализация функции ценности
    V = np.zeros(n_states)
    
    while True:
        delta = 0.0
        for s in range(n_states):
            V_old = V[s]
            # Q(s, a) = R(s, a) + \gamma * \sum_{s'} p(s' | s, a) * V(s')
            Q_s = R[s] + gamma * (P[s] @ V)
            
            # Обновление функции ценности (in-place)
            V[s] = np.max(Q_s)
            delta = max(delta, abs(V_old - V[s]))
            
        # Критерий остановки
        if delta < epsilon:
            break
            
    # Извлечение жадной стратегии \pi(s) = argmax_a Q(s, a)
    pi = np.zeros(n_states, dtype=int)
    for s in range(n_states):
        Q_s = R[s] + gamma * (P[s] @ V)
        pi[s] = np.argmax(Q_s)
        
    return V, pi
```

#### Задача GridWorld, детерминированная сетка $3 \times 3$
Рассмотрим среду с состояниями $(i,j)$, где $i,j \in \{0,1,2\}$.
* Клетка $(2,2)$ является терминальным целевым состоянием. Попадание в нее дает награду $r = 1$.
* Клетка $(1,1)$ недостижима (🚧). 
* Награда за шаг, кроме шага в терминальное состояние, составляет $r = -0.04$. 
* Действия: $\{\uparrow, \downarrow, \leftarrow, \rightarrow\}$. Движение за границу или в клетку $(1,1)$ оставляет агента на месте.  
* Параметр $\gamma = 0.9$.
 
В начале инициализируем $V$-функцию нулями:
 
| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** |
|:---:|:---:|:---:|:---:|
| **2** | $0$ | $0$ | <span style="color: green;">$0$</span> |
| **1** | $0$ | 🚧 | $0$ |
| **0** | $0$ | $0$ | $0$ |
 
Итерация $k=1$. Относительно клетки $(2,1)$ действие «вправо» даёт награду $1$. Итого: $V(2,1) = 1 + 0.9 \cdot V(2,2) = 1$. Действие «влево» дает лишь награду за сделанный шаг: $V(2,1) = -0.04$, следовательно, оставляем первое значение. Аналогично $V(1,2) = 1$. В остальных состояниях агент получает награду за сделанный шаг $-0.04$.

| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** |
|:---:|:---:|:---:|:---:|
| **2** | $-0.04$ | $1$ | <span style="color: green;">$0$</span> |
| **1** | $-0.04$ | 🚧 | $1$ |
| **0** | $-0.04$ | $-0.04$ | $-0.04$ |
 
Итерация $k=2$. Относительно клетки $(2,0)$ действие «вправо» ведёт в клетку $(2,1)$. Получаем $V(2,0) = -0.04 + 0.9 \cdot 1 = 0.86$. Аналогично $V(0,2) = 0.86$. Для клеток $(1,0)$, $(0,1)$, $(0,0)$ награда $V(1,0) = V(0,1) = V(0,0) = -0.04 - 0.9 \cdot 0.04 \approx -0.08$.

| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** |
|:---:|:---:|:---:|:---:|
| **2** | $0.86$ | $1$ | <span style="color: green;">$0$</span> |
| **1** | $-0.08$ | 🚧 | $1$ |
| **0** | $-0.08$ | $-0.08$ | $0.86$ |
 
Итерация $k=3$. Аналогично получаем, что $V(1,0) = V(0,1) = -0.04 + 0.9 \cdot 0.86 \approx 0.73$, $V(0,0) \approx -0.04 - 0.9 \cdot 0.08 \approx -0.11$.

| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** |
|:---:|:---:|:---:|:---:|
| **2** | $0.86$ | $1$ | <span style="color: green;">$0$</span> |
| **1** | $0.73$ | 🚧 | $1$ |
| **0** | $-0.11$ | $0.73$ | $0.82$ |
 
Итерация $k=4$.

| $i \downarrow / j \rightarrow$ | 0 | 1 | 2 |
|:---:|:---:|:---:|:---:|
| **2** | $0.86$ | $1$ | <span style="color: green;">$0$</span> |
| **1** | $0.73$ | 🚧 | $1$ |
| **0** | $0.62$ | $0.73$ | $0.86$ |

Оптимальная политика — выполнение действий по направлению роста $V$-функции.

#### Задача GridWorld, стохастическая сетка $4 \times 4$

Рассмотрим среду с состояниями $(i,j)$, где $i,j \in \{0,1,2,3\}$.
* Клетка $(3,3)$ является целевым терминальным состоянием. Попадание в нее дает награду $r=1$.
* Клетка $(0,3)$ является проигрышным терминальным состоянием (ямой). Попадание в нее дает награду $r=-1$.
* Клетки $(1,1)$ и $(2,2)$ недостижимы (🚧).
* Награда за шаг, кроме шагов в терминальные состояние, составляет $r = -0.04$. 
* Действия: $\{\uparrow, \downarrow, \leftarrow, \rightarrow\}$. Движение за границу или в клетки $(1,1)$ и $(2,2)$ оставляют агента на месте.  
* Стохастичность движения (ветер): агент движется в выбранном направлении с вероятностью $0.8$, вероятность переместиться влево составляет $0.1$, вероятность переместиться вправо составляет $0.1$.
* Параметр $\gamma = 0.9$.

В начале инициализируем $V$-функцию нулями:

| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** | **3** |
|:---:|:---:|:---:|:---:|:---:|
| **3** | $0$ | $0$ | $0$ | <span style="color: green;">$0$</span> |
| **2** | $0$ | $0$ | 🚧 | $0$ |
| **1** | $0$ | 🚧 | $0$ | $0$ |
| **0** | $0$ | $0$ | $0$ | <span style="color: red;">$0$</span> |

Итерация $k=1$. Из клетки $(3,2)$ действие «вправо» с вероятностью $0.8$ даёт награду $1$, с вероятностью $0.2$ агент останется на месте. Ожидаемое значение: $0.8 \cdot 1 - 0.2 \cdot 0.04 \approx 0.79$. Оставшиеся действия дадут меньшее значение, следовательно $V(3,2) \approx 0.79$. Аналогично $V(2,3) \approx 0.79$. Из клетки $(1,3)$ действие «вниз» дает ожидаемое значение $-0.8 \cdot 1 - 0.2 \cdot 0.04 \approx -0.8$, действие «влево» дает ожидаемое значение $-0.8 \cdot 0.04 - 0.1 \cdot 1 - 0.1 \cdot 0.04 \approx -0.14$, «вправо» так же дает примерно $-0.14$, «вверх» дает $-0.04$, следовательно $V(1,3) = -0.04$.

| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** | **3** |
|:---:|:---:|:---:|:---:|:---:|
| **3** | $-0.04$ | $-0.04$ | $0.79$ | <span style="color: green;">$0$</span> |
| **2** | $-0.04$ | $-0.04$ | 🚧 | $0.79$ |
| **1** | $-0.04$ | 🚧 | $-0.04$ | $-0.04$ |
| **0** | $-0.04$ | $-0.04$ | $-0.04$ | <span style="color: red;">$0$</span> |

Итерация $k=3$.

| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** | **3** |
|:---:|:---:|:---:|:---:|:---:|  
| **3** | $0.32$ | $0.67$ | $0.96$ | <span style="color: green;">$0$</span> |  
| **2** | $-0.11$ | $0.32$ | 🚧 | $0.96$ |  
| **1** | $-0.11$ | 🚧 | $0.32$ | $0.67$ |  
| **0** | $-0.11$ | $-0.11$ | $-0.11$ | <span style="color: red;">$0$</span> |

Итерация $k=10$.

| $i \downarrow / j \rightarrow$ | **0** | **1** | **2** | **3** |
|:---:|:---:|:---:|:---:|:---:|  
| **3** | $0.62$ | $0.78$ | $0.97$ | <span style="color: green;">$0$</span> |  
| **2** | $0.51$ | $0.62$ | 🚧 | $0.97$ |  
| **1** | $0.39$ | 🚧 | $0.60$ | $0.78$ |  
| **0** | $0.28$ | $0.22$ | $0.32$ | <span style="color: red;">$0$</span> |

#### Итерация по стратегиям (Policy Iteration)

```python
import numpy as np

def policy_iteration(P, R, gamma, epsilon=1e-4):
    """
    Итерация по стратегиям (Policy Iteration) для конечного МППР.
    
    Параметры:
    - P: np.ndarray формы (|S|, |A|, |S|) — вероятности переходов P[s, a, s'] = p(s' | s, a)
    - R: np.ndarray формы (|S|, |A|) — ожидаемая награда R[s, a] = R(s, a)
    - gamma: float — коэффициент дисконтирования
    - epsilon: float — порог сходимости для этапа Policy Evaluation
    
    Возвращает:
    - V: np.ndarray длины |S| — приближение оптимальной функции ценности V*
    - pi: np.ndarray длины |S| — оптимальная детерминированная стратегия \pi*(s)
    """
    n_states, n_actions, _ = P.shape
    
    # Инициализация произвольной детерминированной стратегии и функции ценности
    pi = np.zeros(n_states, dtype=int)
    V = np.zeros(n_states)
    
    while True:
        # 1. Оценка стратегии (Policy Evaluation, in-place)
        while True:
            delta = 0.0
            for s in range(n_states):
                V_old = V[s]
                a = pi[s]
                # V^{\pi}(s) = R(s, \pi(s)) + \gamma * \sum_{s'} p(s' | s, \pi(s)) * V(s')
                V[s] = R[s, a] + gamma * (P[s, a] @ V)
                delta = max(delta, abs(V_old - V[s]))
                
            if delta < epsilon:
                break
        
        # 2. Улучшение стратегии (Policy Improvement)
        policy_stable = True
        for s in range(n_states):
            old_action = pi[s]
            # Q(s, a) = R(s, a) + \gamma * \sum_{s'} p(s' | s, a) * V(s')
            Q_s = R[s] + gamma * (P[s] @ V)
            
            # Извлечение жадного действия \pi(s) = argmax_a Q(s, a)
            pi[s] = np.argmax(Q_s)
            
            if old_action != pi[s]:
                policy_stable = False
        
        # Критерий остановки: стратегия не изменилась ни в одном состоянии
        if policy_stable:
            break
            
    return V, pi
```

#### Задача GridWorld, детерминированная сетка $2 \times 2$

Рассмотрим среду с состояниями $(i,j)$, где $i,j \in \{0,1\}$.
* Клетка $(1,1)$ является терминальным целевым состоянием. Попадание в неё даёт награду $r=1$.
* Награда за шаг, кроме шага в терминальное состояние, составляет $r = -0.04$. 
* Действия: $\{\uparrow, \downarrow, \leftarrow, \rightarrow\}$. Движение за границу сетки оставляет агента на месте.
* Параметр $\gamma = 0.9$.

В начале зададим произвольную политику $\pi_0$. Пусть эта политика зацикливает агента между клетками $(0,0)$ и $(0,1)$, не позволяя достичь цели:

| $i \downarrow / j \rightarrow$ | **0** | **1** |
|:---:|:---:|:---:|
| **1** | $\rightarrow$ | <span style="color: green;">GOAL</span> |
| **0** | $\rightarrow$ | $\leftarrow$ |

**Итерация $k=1$. Оценка политики.** Для фиксированной политики $\pi_0$ вычислим $V^{\pi_0}$. Для клеток $(0,0)$ и $(0,1)$ образуется замкнутый цикл, поэтому их ценности удовлетворяют системе:
$$
V^{\pi_0}(0,0) = -0.04 + 0.9 V^{\pi_0}(0,1), \quad V^{\pi_0}(0,1) = -0.04 + 0.9 V^{\pi_0}(0,0).
$$
Решая, получаем $V^{\pi_0}(0,0) = V^{\pi_0}(0,1) = -0.4$. Для клетки $(1,0)$ действие «вправо» ведёт непосредственно в цель:
$$
V^{\pi_0}(1,0) = 1 + 0.9 \cdot V^{\pi_0}(1,1).
$$
Учитывая, что в терминальном состоянии $V^{\pi_0}(1,1) = 0$, получаем  $V^{\pi_0}(1,0) = 1$.

| $i \downarrow / j \rightarrow$ | **0** | **1** |
|:---:|:---:|:---:|
| **1** | $1$ | <span style="color: green;">$0$</span> |
| **0** | $-0.40$ | $-0.40$ |

**Итерация $k=1$. Улучшение политики.** Вычисляем $Q$-функции для альтернативных действий. В клетке $(0,0)$ действие «вверх» даёт
$$
Q^{\pi_0}((0,0),\uparrow) = -0.04 + 0.9 \cdot 1 = 0.86,
$$
что превышает текущее значение $-0.4$. В клетке $(0,1)$ действие «вверх» даёт
$$
Q^{\pi_0}((0,1),\uparrow) = 1 + 0.9 \cdot 0 = 1,
$$
что также превышает текущее значение. Новая политика $\pi_1$:

| $i \downarrow / j \rightarrow$ | **0** | **1** |
|:---:|:---:|:---:|
| **1** | $\rightarrow$ | <span style="color: green;">GOAL</span> |
| **0** | $\uparrow$ | $\uparrow$ |

**Итерация $k=2$. Оценка политики.** При новой политике из клеток $(0,1)$ и $(1,0)$ агент попадает в цель за один шаг: $V^{\pi_1}(1,0) = V^{\pi_1}(0,1) = 1$. Из $(0,0)$ агент переходит в $(1,0)$:
$$
V^{\pi_1}(0,0) = -0.04 + 0.9 \cdot 1 = 0.86.
$$

| $i \downarrow / j \rightarrow$ | **0** | **1** |
|:---:|:---:|:---:|
| **1** | $1$ | <span style="color: green;">$0$</span> |
| **0** | $0.86$ | $1$ |

**Итерация $k=2$. Улучшение политики.** Проверка показывает, что для всех состояний текущие действия остаются оптимальными. Заметим, что переход из клетки $(0,0)$ в клетку $(0,1)$ оставляет ценность состояния прежним. Поскольку новая политика $\pi_2$ совпадает с $\pi_1$, алгоритм останавливается — мы достигли оптимальной политики.

#### От динамического программирования к обучению без модели

Ограничения методов динамического программирования:

1. **Требование полной модели среды (Model-Based)**
   Необходимо априорное знание ядра переходов $p(s', r \mid s, a)$. В большинстве реальных прикладных задач (робототехника, беспилотное вождение, видеоигры) модель динамики неизвестна или или слишком сложна для точного вычисления.
2. **Проклятие размерности (Curse of Dimensionality)**
   Алгоритмы ДП требуют хранения вектора ценностей размера $|\mathcal{S}|$ и матрицы переходов размера $|\mathcal{S}| |\mathcal{A}| \times |\mathcal{S}|$. Если состояние описывается вектором из $d$ параметров, где каждая компонента вектора принимает всего $K$ значений, то мощность пространства состояний растёт экспоненциально: $|\mathcal{S}| = K^d$. Например, для системы всего с $d = 8$ датчиками при грубой сетке из $K = 10$ значений на каждый датчик размер пространства составляет $|\mathcal{S}| = 10^8$ состояний. Хранение только одной функции ценности $V(s)$ в формате `float64` потребует $\approx 800$ МБ, а хранение плотной матрицы переходов $P(s' \mid s, a)$ даже для $| \mathcal{A} | = 4$ действий потребовало бы порядка $4 \times 10^{16}$ чисел ($\approx 320$ петабайт оперативной памяти), что делает табличные методы ДП принципиально неприменимыми для многомерных задач.
3. **Необходимость полных развёрток (Full Sweeps):**  
   На каждом шаге алгоритмы ДП обязаны обновлять оценки для абсолютно всех состояний $s \in \mathcal{S}$, даже если большинство из них реально не посещается.

Преодоление этих ограничений привело к развитию **обучения с подкреплением без модели (Model-Free RL)**, где агент оптимизирует поведение исключительно на основе опыта взаимодействия со средой. Этот переход основан на нескольких ключевых концепциях:

* **Выборочные обновления (Sample Backups) вместо полных (Full Backups)**
   Математические ожидания по всем возможным переходам заменяются стохастическими обновлениями по единичным реально наблюдаемым кортежам $(s_t, a_t, r_{t+1}, s_{t+1})$.
* **Бутстрэппинг (Bootstrapping)**
   Обновление текущей оценки ценности $V(s_t)$ или $Q(s_t, a_t)$ производится с использованием прогнозируемой ценности последующего состояния $V(s_{t+1})$ или $Q(s_{t+1}, a_{t+1})$ без необходимости ожидания финала траектории.
* **Компромисс исследования и эксплуатации (Exploration vs Exploitation)**
   Поскольку агент сам генерирует обучающие данные своими действиями, он вынужден балансировать между получением максимальной известной награды (эксплуатация) и сбором новой информации о среде (исследование).
* **Функциональная аппроксимация**
   Для преодоления проклятия размерности применяется функциональная аппроксимация (например, нейронные сети вместо таблиц), которая позволяет обобщать знания на непосещённые состояния и работать с непрерывными или высокоразмерными пространствами.